# **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
# Limit to single thread to avoid conflicts during hyperparameter tuning
# os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import numpy as np
import optuna
import implicit

from Challenge.paths import load_cv_folds
from Challenge.hyper_tuning import ModelOptimizer
from Challenge.utils import evaluate_recommender_implicit

Running on local — storage at: /home/luigi/RecSys


/home/luigi/RecSys/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/home/luigi/RecSys/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


# **Load Data**

In [4]:
# Load datasets
folds = load_cv_folds(k=5)

In [ ]:
# Initialize optimizer
optimizer = ModelOptimizer("IALS")

## **Hyperparameter search**

In [ ]:
STUDY_NAME = "IALS_implicit_optimization"

In [ ]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "factors": optuna_trial.suggest_categorical("factors", [32, 64, 128]),
        "regularization": optuna_trial.suggest_float("regularization", 0.01, 0.1),
        "alpha": optuna_trial.suggest_float("alpha", 1.0, 40.0),
        "iterations": 15 # Fixed for speed
    }

    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = implicit.cpu.als.AlternatingLeastSquares(
            factors=params["factors"],
            regularization=params["regularization"],
            alpha=params["alpha"],
            iterations=params["iterations"],
            num_threads=0,          # 0 = Use all CPU cores
            random_state=42
        )

        recommender_instance.fit(URM_train)
        
        # Evaluate
        score = evaluate_recommender_implicit(
            recommender_instance,
            20,
            URM_train,
            URM_validation
        )
        
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [7]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=50
)

[I 2025-11-28 13:45:34,862] Using an existing study with name 'IALS_implicit_optimization' instead of creating a new one.


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

Eval Batches: 100%|██████████| 28/28 [00:03<00:00,  8.35it/s]

  Fold 1/5 - Score: 0.24379966545913181


...

Eval Batches: 100%|██████████| 28/28 [00:02<00:00, 11.30it/s]

  Fold 3/5 - Score: 0.22746546470802176


  0%|          | 0/15 [00:00<?, ?it/s]

Eval Batches: 100%|██████████| 28/28 [00:02<00:00, 11.28it/s]


  Fold 4/5 - Score: 0.22634492764273012
[I 2025-11-28 14:05:02,626] Trial 95 finished with value: 0.22691701202414044 and parameters: {'factors': 128, 'regularization': 0.08900303636215784, 'alpha': 39.163882000237464}. Best is trial 68 with value: 0.24322776511452449.

Study statistics: 
  Number of finished trials:  96
  Number of pruned trials:  0
  Number of complete trials:  91

Best Value: 0.24322776511452449
Best Params: {'factors': 128, 'regularization': 0.09986370128000664, 'alpha': 11.650900056856798}


In [14]:
optuna_study.best_value, optuna_study.best_params

(0.24322776511452449,
 {'factors': 128,
  'regularization': 0.09986370128000664,
  'alpha': 11.650900056856798})

In [8]:
optuna.visualization.plot_optimization_history(optuna_study)

In [9]:
optuna.visualization.plot_param_importances(optuna_study)

In [10]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Extend ranges**

In [15]:
STUDY_NAME = "IALS_implicit_optimization_v2"

In [16]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "factors": optuna_trial.suggest_int("factors", 128, 512, step=32),
        "regularization": optuna_trial.suggest_float("regularization", 0.01, 1.0, log=True),
        "alpha": optuna_trial.suggest_float("alpha", 5.0, 20.0),
        "iterations": 15, # Fixed for speed
        "num_threads": 0  # Use all CPU cores
    }

    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = implicit.cpu.als.AlternatingLeastSquares(
            factors=params["factors"],
            regularization=params["regularization"],
            alpha=params["alpha"],
            iterations=params["iterations"],
            num_threads=params["num_threads"],         
            random_state=42
        )

        recommender_instance.fit(URM_train, show_progress=False)
        
        # Evaluate
        score = evaluate_recommender_implicit(
            recommender_instance,
            20,
            URM_train,
            URM_validation
        )
        
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [17]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=50
)

[I 2025-11-28 14:24:44,846] A new study created in RDB with name: IALS_implicit_optimization_v2


  0%|          | 0/50 [00:00<?, ?it/s]

Eval Batches: 100%|██████████| 28/28 [00:02<00:00,  9.50it/s]


  Fold 1/5 - Score: 0.21233080132359572


Eval Batches: 100%|██████████| 28/28 [00:02<00:00, 10.08it/s]


...

  Fold 2/5 - Score: 0.2294814314766356


Eval Batches: 100%|██████████| 28/28 [00:02<00:00, 10.10it/s]


  Fold 3/5 - Score: 0.22949384422448305


Eval Batches: 100%|██████████| 28/28 [00:02<00:00, 10.57it/s]


  Fold 4/5 - Score: 0.23001467505666018
[I 2025-11-28 14:56:24,123] Trial 49 finished with value: 0.22965337004038958 and parameters: {'factors': 256, 'regularization': 0.20431995508148146, 'alpha': 9.122667055537093}. Best is trial 31 with value: 0.2441437125140828.

Study statistics: 
  Number of finished trials:  50
  Number of pruned trials:  0
  Number of complete trials:  50

Best Value: 0.2441437125140828
Best Params: {'factors': 128, 'regularization': 0.37966297655317477, 'alpha': 11.549619120618086}


In [18]:
optuna.visualization.plot_optimization_history(optuna_study)

In [19]:
optuna.visualization.plot_param_importances(optuna_study)

In [20]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **More focused ranges**

In [21]:
STUDY_NAME = "IALS_implicit_optimization_v3"

In [23]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "factors": optuna_trial.suggest_int("factors", 96, 160),
        "regularization": optuna_trial.suggest_float("regularization", 0.01, 1.0, log=True),
        "alpha": optuna_trial.suggest_float("alpha", 5.0, 20.0),
        "iterations": 15, # Fixed for speed
        "num_threads": 0  # Use all CPU cores
    }

    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = implicit.cpu.als.AlternatingLeastSquares(
            factors=params["factors"],
            regularization=params["regularization"],
            alpha=params["alpha"],
            iterations=params["iterations"],
            num_threads=params["num_threads"],         
            random_state=42
        )

        recommender_instance.fit(URM_train, show_progress=False)
        
        # Evaluate
        score = evaluate_recommender_implicit(
            recommender_instance,
            20,
            URM_train,
            URM_validation
        )
        
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [24]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=50
)

[I 2025-11-28 15:01:46,759] A new study created in RDB with name: IALS_implicit_optimization_v3


  0%|          | 0/50 [00:00<?, ?it/s]

Eval Batches: 100%|██████████| 28/28 [00:03<00:00,  9.11it/s]


  Fold 1/5 - Score: 0.23811124733344713


Eval Batches: 100%|██████████| 28/28 [00:03<00:00,  8.85it/s]


...

  Fold 2/5 - Score: 0.24606690756816024


Eval Batches: 100%|██████████| 28/28 [00:02<00:00,  9.76it/s]


  Fold 3/5 - Score: 0.24577287852578503


Eval Batches: 100%|██████████| 28/28 [00:02<00:00, 10.18it/s]


  Fold 4/5 - Score: 0.24423608003870098
[I 2025-11-28 15:22:42,689] Trial 49 finished with value: 0.245361016701327 and parameters: {'factors': 104, 'regularization': 0.7606616820747529, 'alpha': 10.2378092017718}. Best is trial 25 with value: 0.24631511017779828.

Study statistics: 
  Number of finished trials:  50
  Number of pruned trials:  0
  Number of complete trials:  50

Best Value: 0.24631511017779828
Best Params: {'factors': 100, 'regularization': 0.5617610048419468, 'alpha': 11.108713010542706}


In [25]:
optuna.visualization.plot_optimization_history(optuna_study)

In [26]:
optuna.visualization.plot_param_importances(optuna_study)

In [27]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Number of iteratons**

In [28]:
STUDY_NAME = "IALS_implicit_optimization_v3_iterations"

In [29]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "factors": 100,
        "regularization": 0.5617610048419468,
        "alpha": 11.108713010542706,
        "iterations": optuna_trial.suggest_int("iterations", 10, 50),
        "num_threads": 0  # Use all CPU cores
    }

    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = implicit.cpu.als.AlternatingLeastSquares(
            factors=params["factors"],
            regularization=params["regularization"],
            alpha=params["alpha"],
            iterations=params["iterations"],
            num_threads=params["num_threads"],         
            random_state=42
        )

        recommender_instance.fit(URM_train, show_progress=False)
        
        # Evaluate
        score = evaluate_recommender_implicit(
            recommender_instance,
            20,
            URM_train,
            URM_validation
        )
        
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [30]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=15
)

[I 2025-11-28 15:26:10,190] A new study created in RDB with name: IALS_implicit_optimization_v3_iterations


  0%|          | 0/15 [00:00<?, ?it/s]

Eval Batches: 100%|██████████| 28/28 [00:02<00:00,  9.56it/s]


  Fold 1/5 - Score: 0.2453027811472317


Eval Batches: 100%|██████████| 28/28 [00:02<00:00,  9.42it/s]


...

  Fold 3/5 - Score: 0.24710570983330424


Eval Batches: 100%|██████████| 28/28 [00:02<00:00,  9.41it/s]


  Fold 4/5 - Score: 0.24488107243136892


Eval Batches: 100%|██████████| 28/28 [00:02<00:00,  9.86it/s]


  Fold 5/5 - Score: 0.24758857339323986
[I 2025-11-28 15:34:57,179] Trial 14 finished with value: 0.2466533703635685 and parameters: {'iterations': 29}. Best is trial 5 with value: 0.24669865029393198.

Study statistics: 
  Number of finished trials:  15
  Number of pruned trials:  0
  Number of complete trials:  15

Best Value: 0.24669865029393198
Best Params: {'iterations': 31}


In [31]:
optuna.visualization.plot_optimization_history(optuna_study)

In [32]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- First Search:
0.24322776511452449,
 {'factors': 128,
  'regularization': 0.09986370128000664,
  'alpha': 11.650900056856798}

- Second Search:
Best Value: 0.2441437125140828
Best Params: {'factors': 128, 'regularization': 0.37966297655317477, 'alpha': 11.549619120618086}

- Third Search:
Best Value: 0.24631511017779828
Best Params: {'factors': 100, 'regularization': 0.5617610048419468, 'alpha': 11.108713010542706}

- Final Search:
Best Value: 0.24669865029393198
Best Params: {'factors': 100, 'regularization': 0.5617610048419468, 'alpha': 11.108713010542706, 'iterations': 31}
